# 🌐 Tu primer sitio web con Flask: de cero a producción

En este cuaderno construirás **paso a paso** un pequeño sitio web con Python y [Flask](https://flask.palletsprojects.com/): dos páginas (*Inicio* y *Acerca de*), un menú de navegación, estilos CSS y, al final, lo publicarás en internet.

**Cómo usarlo:** ejecuta las celdas en orden (`Shift + Enter`). Las celdas que empiezan por `%%writefile` **crean los archivos reales del proyecto** dentro de la carpeta `mi_sitio_flask/`. Esa carpeta es la que subirás al servidor.

**Requisitos:** Python 3.10 o superior y nociones básicas de HTML. No necesitas saber nada de servidores.

### Mapa del cuaderno

| Paso | Qué haremos |
|---|---|
| 0 | Entender cómo funciona una web con Flask |
| 1 | Preparar el entorno y la estructura de carpetas |
| 2 | Nuestra primera aplicación (una sola ruta) |
| 3 | Devolver plantillas HTML |
| 4 | Plantilla base reutilizable (herencia con Jinja) |
| 5 | Menú de navegación |
| 6 | Estilos CSS |
| 7 | Página de error 404 |
| 8 | Entorno virtual y `requirements.txt` |
| 9 | Publicar en un servidor |
| 10 | Mantener y actualizar la web |

## Paso 0 · ¿Cómo funciona una web hecha con Flask?

Cada vez que escribes una dirección en el navegador ocurre esto:

```
 Navegador ──(1) petición: GET /about/ ──▶  Flask
                                              │ (2) busca la "ruta" /about/
                                              │ (3) ejecuta la función Python asociada
                                              │ (4) rellena una plantilla HTML
 Navegador ◀──(5) respuesta: página HTML ─────┘
```

Tres ideas clave:

- **Ruta**: asocia una dirección (`/about/`) con una función de Python.
- **Plantilla**: un archivo HTML con "huecos" que Flask rellena (se escribe en el lenguaje **Jinja**).
- **Archivos estáticos**: CSS, imágenes y JavaScript que se envían tal cual.

## Paso 1 · Preparar el entorno y la estructura de carpetas

Instalamos Flask (en el paso 8 aprenderás a hacerlo en un entorno virtual aislado, que es la buena práctica para tus proyectos reales).

In [ ]:
%pip install -q "flask>=3.0"

import sys
from importlib.metadata import version
print("Python:", sys.version.split()[0])
print("Flask :", version("flask"))

Flask espera encontrar **dos carpetas con nombre fijo**: `templates/` para los HTML y `static/` para CSS, imágenes y JS.

In [ ]:
from pathlib import Path

PROYECTO = Path("mi_sitio_flask")
for carpeta in ["templates", "static/css"]:
    (PROYECTO / carpeta).mkdir(parents=True, exist_ok=True)

for ruta in sorted(PROYECTO.rglob("*")):
    print(ruta)

### 🛠️ Un pequeño ayudante para este cuaderno

Normalmente arrancarías el servidor desde la terminal. Para poder **ver el resultado sin salir del cuaderno**, definimos dos funciones auxiliares que usan el *cliente de pruebas* de Flask (simula a un navegador). No necesitas entender su código ahora.

In [ ]:
import importlib, re
from html import escape
from IPython.display import HTML, display

sys.dont_write_bytecode = True           # evita cachés al reescribir app.py
sys.path.insert(0, str(PROYECTO.resolve()))

def cargar_app():
    """Importa (o recarga) app.py y devuelve la aplicación Flask."""
    import app as modulo
    importlib.reload(modulo)
    return modulo.app

def probar(ruta="/"):
    """Hace una petición GET y muestra el código de estado y el HTML recibido."""
    respuesta = cargar_app().test_client().get(ruta)
    print(f"GET {ruta}  →  código {respuesta.status_code}\n")
    print(respuesta.get_data(as_text=True))

def ver_pagina(ruta="/", alto=260):
    """Muestra la página renderizada (con su CSS) dentro del cuaderno."""
    pagina = cargar_app().test_client().get(ruta).get_data(as_text=True)
    css = PROYECTO / "static/css/main.css"
    if css.exists():   # incrustamos el CSS porque el iframe no puede pedirlo al servidor
        pagina = re.sub(r"<link[^>]*main\.css[^>]*>",
                        lambda m: f"<style>{css.read_text(encoding='utf-8')}</style>", pagina)
    display(HTML(f'<iframe srcdoc="{escape(pagina, quote=True)}" width="100%" '
                 f'height="{alto}" style="border:1px solid #bbb;border-radius:6px"></iframe>'))

## Paso 2 · Nuestra primera aplicación

Creamos `app.py`. Fíjate en las tres piezas: crear la aplicación, definir una **ruta** con el decorador `@app.route` y devolver una respuesta.

> 📝 *Mejora:* el archivo original se llamaba `script1.py`. Lo llamamos `app.py` porque es el nombre que Flask busca por defecto.

In [ ]:
%%writefile mi_sitio_flask/app.py
from flask import Flask

app = Flask(__name__)


@app.route("/")
def home():
    return "<h1>¡Hola desde Flask!</h1>"

In [ ]:
probar("/")

### ▶️ Ejecutarlo de verdad (desde una terminal)

Abre una terminal **dentro de la carpeta `mi_sitio_flask`** y ejecuta:

```bash
flask run --debug
```

Abre <http://127.0.0.1:5000> en el navegador. El modo `--debug` recarga el servidor cada vez que guardas un cambio y muestra errores detallados. Para parar el servidor: `Ctrl + C`.

> ⚠️ El modo *debug* es **solo para desarrollo**: nunca debe activarse en un servidor público.

## Paso 3 · Devolver plantillas HTML

Escribir HTML dentro de strings de Python no escala. Lo correcto es guardarlo en archivos dentro de `templates/` y devolverlo con `render_template`. Añadimos también la segunda página.

In [ ]:
%%writefile mi_sitio_flask/app.py
from flask import Flask, render_template

app = Flask(__name__)


@app.route("/")
def home():
    return render_template("home.html")


@app.route("/about/")
def about():
    return render_template("about.html")

In [ ]:
%%writefile mi_sitio_flask/templates/home.html
<h1>Mi página de inicio</h1>
<p>Esta es una web de prueba hecha con Flask.</p>

In [ ]:
%%writefile mi_sitio_flask/templates/about.html
<h1>Acerca de</h1>
<p>Esta es la página «Acerca de».</p>

In [ ]:
probar("/about/")

## Paso 4 · Plantilla base reutilizable (herencia con Jinja)

Si cada página repite `<html>`, `<head>`, cabecera... tendríamos que cambiarlo todo en cada archivo cada vez que quisiéramos retocar algo. Jinja resuelve esto con **herencia de plantillas**:

- `layout.html` define el esqueleto común y marca con `{% block %}` los huecos que cambian.
- Cada página hace `{% extends "layout.html" %}` y rellena solo sus huecos.

Además de `content`, creamos un bloque `title` para que cada página tenga su propio título en la pestaña del navegador.

> 📝 *Mejoras adicionales:* añadimos `lang="es"`, `charset` y `viewport` (imprescindibles para acentos y para verse bien en móvil), y usamos `<main>` para el contenido principal.

In [ ]:
%%writefile mi_sitio_flask/templates/layout.html
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{% block title %}Mi sitio Flask{% endblock %}</title>
</head>
<body>
  <header>
    <div class="container">
      <span class="logo">Mi aplicación web</span>
    </div>
  </header>

  <main class="container">
    {% block content %}{% endblock %}
  </main>
</body>
</html>

In [ ]:
%%writefile mi_sitio_flask/templates/home.html
{% extends "layout.html" %}

{% block title %}Inicio · Mi sitio{% endblock %}

{% block content %}
<section class="panel">
  <h1>Mi página de inicio</h1>
  <p>Esta es una web de prueba hecha con Flask.</p>
</section>
{% endblock %}

In [ ]:
%%writefile mi_sitio_flask/templates/about.html
{% extends "layout.html" %}

{% block title %}Acerca de · Mi sitio{% endblock %}

{% block content %}
<section class="panel">
  <h1>Acerca de</h1>
  <p>Esta es la página «Acerca de».</p>
</section>
{% endblock %}

In [ ]:
ver_pagina("/")

## Paso 5 · Menú de navegación

Añadimos un `<nav>` a la plantilla base: así aparecerá automáticamente en todas las páginas.

Dos detalles importantes:

- **`url_for('home')`** genera la dirección a partir del *nombre de la función*. Si algún día cambias `/about/` por `/sobre-mi/`, los enlaces se actualizan solos.
- **`request.endpoint`** nos dice en qué página estamos, y lo usamos para marcar el enlace activo.

In [ ]:
%%writefile mi_sitio_flask/templates/layout.html
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{% block title %}Mi sitio Flask{% endblock %}</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='css/main.css') }}">
</head>
<body>
  <header>
    <div class="container barra">
      <a class="logo" href="{{ url_for('home') }}">Mi aplicación web</a>
      <nav>
        <ul class="menu">
          <li><a href="{{ url_for('home') }}" class="{{ 'activo' if request.endpoint == 'home' }}">Inicio</a></li>
          <li><a href="{{ url_for('about') }}" class="{{ 'activo' if request.endpoint == 'about' }}">Acerca de</a></li>
        </ul>
      </nav>
    </div>
  </header>

  <main class="container">
    {% block content %}{% endblock %}
  </main>
</body>
</html>

## Paso 6 · Estilos CSS

El CSS va en `static/css/main.css`. La plantilla lo enlaza con `url_for('static', filename=...)`, que genera la ruta correcta.

In [ ]:
%%writefile mi_sitio_flask/static/css/main.css
:root {
  --color-cabecera: #DFB887;
  --color-panel:    #E6E6FA;
  --color-texto:    #006600;
  --color-logo:     #3b2410;
  --color-enlace:   #444;
}

*, *::before, *::after { box-sizing: border-box; }

body {
  margin: 0;
  font-family: system-ui, "Helvetica Neue", Helvetica, Arial, sans-serif;
  line-height: 1.6;
  color: var(--color-texto);
}

/* Centra el contenido y limita su ancho */
.container {
  max-width: 1200px;
  margin: 0 auto;
  padding: 0 1rem;
}

/* Cabecera */
header {
  background-color: var(--color-cabecera);
  margin-bottom: 1rem;
}

.barra {
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  justify-content: space-between;
  gap: .5rem 2rem;
  min-height: 56px;
}

.logo {
  font-size: 1.5rem;
  font-weight: 700;
  text-transform: uppercase;
  color: var(--color-logo);
  text-decoration: none;
}

/* Menú */
.menu {
  display: flex;
  gap: 2rem;
  margin: 0;
  padding: 0;
  list-style: none;
}

.menu a {
  color: var(--color-enlace);
  font-weight: 700;
  text-decoration: none;
}

.menu a:hover,
.menu a.activo {
  color: #000;
  text-decoration: underline;
}

/* Contenido */
.panel {
  padding: .5rem 1.5rem 1.5rem;
  background-color: var(--color-panel);
  border-radius: 6px;
}

In [ ]:
ver_pagina("/about/")

> 💡 **Si cambias el CSS y no ves el cambio** en tu navegador, es la caché: recarga forzada con `Ctrl + F5` (o `Cmd + Shift + R` en Mac).

## Paso 7 · Página de error 404

¿Qué ve el visitante si escribe una dirección que no existe? Ahora mismo, un error feo y genérico. Con `@app.errorhandler(404)` mostramos una página propia, con el mismo diseño del sitio. Fíjate en el **`, 404`**: es el código de estado HTTP que acompaña a la respuesta.

In [ ]:
%%writefile mi_sitio_flask/app.py
from flask import Flask, render_template

app = Flask(__name__)


@app.route("/")
def home():
    return render_template("home.html")


@app.route("/about/")
def about():
    return render_template("about.html")


@app.errorhandler(404)
def pagina_no_encontrada(error):
    return render_template("404.html"), 404


if __name__ == "__main__":
    # Solo para desarrollo. En un servidor real se usa gunicorn (paso 9).
    app.run(debug=True)

In [ ]:
%%writefile mi_sitio_flask/templates/404.html
{% extends "layout.html" %}

{% block title %}Página no encontrada{% endblock %}

{% block content %}
<section class="panel">
  <h1>404 · Página no encontrada</h1>
  <p>La dirección que buscas no existe. <a href="{{ url_for('home') }}">Volver al inicio</a>.</p>
</section>
{% endblock %}

In [ ]:
ver_pagina("/esta-pagina-no-existe/", alto=220)

## Paso 8 · Entorno virtual y `requirements.txt`

Un **entorno virtual** es una instalación de Python aislada para un solo proyecto. Así, las librerías de un proyecto no chocan con las de otro ni con las de tu sistema. Es la buena práctica desde **antes** de escribir la primera línea; aquí lo dejamos para ahora solo por claridad.

Desde una terminal, **dentro de la carpeta del proyecto**:

**Windows (PowerShell)**
```powershell
python -m venv .venv
venv\Scripts\Activate.ps1
```

**macOS / Linux**
```bash
python3 -m venv .venv
source venv/bin/activate
```

Cuando está activo verás `(venv)` al principio de la línea. Ahora instala las dependencias **dentro** del entorno:

```bash
pip install flask gunicorn
```

> `gunicorn` es el servidor de producción que usará la nube; no funciona en Windows, pero instalarlo no da problemas y en el servidor (Linux) se ejecutará correctamente.

Para que el servidor sepa qué instalar, listamos las dependencias en `requirements.txt`.

In [ ]:
%%writefile mi_sitio_flask/requirements.txt
Flask>=3.0,<4
gunicorn>=22.0

> 📝 *Mejora practicada al añadir rangos en las versiones de los paquetes:* en lugar de fijar **todas** las versiones con `pip freeze` (Flask 0.10.1, Jinja2 2.8, Werkzeug 0.11...), aquí indicamos solo las dependencias directas y un rango de versiones; pip instala el resto. Si prefieres fijar versiones exactas, ejecuta `pip freeze > requirements.txt` desde tu entorno virtual y actualízalas de vez en cuando.

## Paso 9 · Publicar el sitio en internet

Hasta ahora solo tú ves la web (`localhost`). Para que la vea cualquiera, hay que subirla a un servidor.

> ⚠️ **Heroku** era una gran opción en el pasado, pero eliminó su plan gratuito en 2022. Usaremos [Render](https://render.com), que a fecha de hoy ofrece un plan gratuito con limitaciones (por ejemplo, la web "se duerme" tras un rato sin visitas y la primera carga tarda más). Otras opciones: PythonAnywhere, Railway o Fly.io. **Los planes gratuitos cambian a menudo: comprueba las condiciones vigentes antes de empezar.**

### 9.1 · Archivos de apoyo

Creamos un `.gitignore` para no subir archivos que sobran (el entorno virtual, cachés, archivos temporales) y un `Procfile` (lo leen plataformas como Heroku; en Render escribirás el comando a mano, pero es inofensivo y útil si cambias de plataforma).

In [ ]:
%%writefile mi_sitio_flask/.gitignore
venv/
.venv/
__pycache__/
*.pyc
*.swp
.env
.ipynb_checkpoints/

In [ ]:
%%writefile mi_sitio_flask/Procfile
web: gunicorn app:app

`gunicorn app:app` significa: *«en el archivo `app.py`, usa la variable `app`»*. 

> 📝 *Ojo:* el `Procfile` debe evitar terminar las líneas con formato de Windows (`\r\n`). Conviene guardarlo con saltos de línea Unix (`\n`), como hace la celda de arriba.

### 9.2 · Subir el código a GitHub

Necesitas [Git](https://git-scm.com/downloads) y una cuenta en GitHub. Crea un repositorio vacío en GitHub y, desde la terminal en la carpeta `mi_sitio_flask`:

```bash
git init
git add .
git commit -m "Primera versión del sitio"
git branch -M main
git remote add origin https://github.com/TU_USUARIO/mi_sitio_flask.git
git push -u origin main
```

### 9.3 · Crear el servicio en Render

1. Crea una cuenta en Render e inicia sesión con GitHub.
2. Pulsa **New + → Web Service** y elige tu repositorio.
3. Rellena la configuración:

| Campo | Valor |
|---|---|
| Language / Runtime | `Python 3` |
| Build Command | `pip install -r requirements.txt` |
| Start Command | `gunicorn app:app` |
| Instance Type | `Free` |

4. En *Environment Variables* añade `PYTHON_VERSION` con la versión completa que usas en local (por ejemplo `3.12.3`); si no, Render usará la suya por defecto.
5. Pulsa **Deploy**. Cuando termine, Render te dará una dirección del tipo `https://tu-proyecto.onrender.com`. ¡Tu web ya es pública! 🎉

> 🔎 **Si algo falla**, lee el *log* de despliegue en Render: casi siempre indica el motivo (una dependencia que falta en `requirements.txt`, un nombre mal escrito en el comando de arranque...).

## Paso 10 · Mantener y actualizar la web

Mantener una web no es vigilarla cada día (la plataforma se encarga de que siga en marcha), sino **hacer cambios**: añadir una página, retocar el CSS, etc. El flujo de trabajo siempre es el mismo:

```
 1. Editar en local  →  2. Probar en local  →  3. Git commit  →  4. Git push  →  5. Render redespliega
```

Ejemplo: añade una línea a `templates/about.html`, por ejemplo `<p>Esto se añadió más tarde.</p>`, y cambia el color del texto en `main.css`. Prueba en local:

```bash
flask run --debug
```

Cuando estés conforme, publica los cambios:

```bash
git add .
git commit -m "Añadida línea en la página Acerca de"
git push
```

Por defecto, Render detecta el *push* y **redespliega solo**. Primero se prueba en *desarrollo* (tu ordenador) y solo después se sube a *producción* (el servidor público).

> 💡 Con `git status` ves qué archivos has modificado y con `git log --oneline` el historial de cambios. Si algo sale mal, siempre puedes volver a una versión anterior.

## 📋 Ejercicios para practicar

1. **Nueva página:** crea `/contacto/` con su plantilla y su enlace en el menú.
2. **Variables:** crea la ruta `/hola/<nombre>/` y muestra `{{ nombre }}` en una plantilla (pista: `def saludo(nombre): return render_template("saludo.html", nombre=nombre)`).
3. **Bucles en Jinja:** pasa una lista de tres cursos desde Python y muéstrala con `{% for curso in cursos %}<li>{{ curso }}</li>{% endfor %}`.
4. **Estilo propio:** cambia la paleta completa modificando solo las variables de `:root` en el CSS.
5. **Publicación:** haz un cambio visible, súbelo con `git push` y comprueba cómo se actualiza la web pública.